In [2]:
# CELL 1 — Phase 5 config and dataset loading
import numpy as np
import torch
import torch.nn as nn
import math
import time
import json
from itertools import combinations
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, matthews_corrcoef
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import combine_pvalues

N_QUBITS = 4
ENTANGLING_LAYERS = 2
D_MODEL = 64
D_FF = 128
N_HEADS = 4
N_TOKENS = 225
TRAIN_BATCH_SIZE = 32
EPOCHS = 50
LR = 2e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SEEDS = [42, 43, 44]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

ABLATION_DATASETS = ["IndianPines", "PaviaUniversity", "Salinas", "KSC", "Botswana"]
datasets = {}
for name in ABLATION_DATASETS:
    d = np.load(f"preprocessed/{name}.npz")
    datasets[name] = {
        "train_tokens": torch.tensor(d["train_tokens"], dtype=torch.float32),
        "train_labels": torch.tensor(d["train_labels"] - 1, dtype=torch.long),
        "val_tokens": torch.tensor(d["val_tokens"], dtype=torch.float32),
        "val_labels": torch.tensor(d["val_labels"] - 1, dtype=torch.long),
        "test_tokens": torch.tensor(d["test_tokens"], dtype=torch.float32),
        "test_labels": torch.tensor(d["test_labels"] - 1, dtype=torch.long),
    }
    datasets[name]["k_dim"] = datasets[name]["train_tokens"].shape[-1]
    datasets[name]["n_classes"] = int(datasets[name]["train_labels"].max().item()) + 1
    print(f"{name}: k={datasets[name]['k_dim']}, classes={datasets[name]['n_classes']}")

Using device: cuda
IndianPines: k=31, classes=16
PaviaUniversity: k=10, classes=9
Salinas: k=10, classes=16
KSC: k=10, classes=13
Botswana: k=10, classes=14


In [3]:
# CELL 2 — Encoder variants for the 5-config ablation suite
import pennylane as qml

QUANTUM_DEVICE_NAME = "default.qubit"
DIFF_METHOD = "backprop"

def build_quantum_layer(n_qubits):
    dev = qml.device(QUANTUM_DEVICE_NAME, wires=n_qubits)
    weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=ENTANGLING_LAYERS, n_wires=n_qubits)
    @qml.qnode(dev, interface="torch", diff_method=DIFF_METHOD)
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(w)) for w in range(n_qubits)]
    return qml.qnn.TorchLayer(circuit, {"weights": weight_shape})


class FullQuantumEncoder(nn.Module):
    """Config: Full — 4-qubit quantum encoder."""
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, N_QUBITS)
        self.q_layer = build_quantum_layer(N_QUBITS)
        self.out_proj = nn.Linear(N_QUBITS, D_MODEL)
    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, N_QUBITS)
        q_out = self.q_layer(flat).reshape(b, n, N_QUBITS)
        return self.out_proj(q_out)


class OneQubitEncoder(nn.Module):
    """Config: 1-Qubit — quantum encoder reduced to a single qubit."""
    def __init__(self, k_dim):
        super().__init__()
        self.angle_proj = nn.Linear(k_dim, 1)
        self.q_layer = build_quantum_layer(1)
        self.out_proj = nn.Linear(1, D_MODEL)
    def forward(self, tokens):
        b, n, k = tokens.shape
        theta = math.pi * torch.tanh(self.angle_proj(tokens))
        flat = theta.reshape(b * n, 1)
        q_out = self.q_layer(flat).reshape(b, n, 1)
        return self.out_proj(q_out)


class ClassicalMirrorEncoder(nn.Module):
    """Config: w/o QNN — classical mirror of the same 4-dim bottleneck shape."""
    def __init__(self, k_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(k_dim, N_QUBITS), nn.Tanh(), nn.Linear(N_QUBITS, D_MODEL))
    def forward(self, tokens):
        return self.net(tokens)


class FullRankEncoder(nn.Module):
    """Config: w/o Bottleneck — full-rank classical, no 4-dim intermediate."""
    def __init__(self, k_dim):
        super().__init__()
        self.net = nn.Linear(k_dim, D_MODEL)
    def forward(self, tokens):
        return self.net(tokens)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, n_tokens, d_model):
        super().__init__()
        pe = torch.zeros(n_tokens, d_model)
        pos = torch.arange(0, n_tokens, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe


ENCODER_REGISTRY = {
    "Full": FullQuantumEncoder,
    "w/o QNN": ClassicalMirrorEncoder,
    "w/o Bottleneck": FullRankEncoder,
    "1-Qubit": OneQubitEncoder,
    "w/o PosEnc": FullQuantumEncoder,  # same encoder as Full; PosEnc removed at model level
}

class AblationQuantFormer(nn.Module):
    def __init__(self, k_dim, n_classes, config_name):
        super().__init__()
        self.config_name = config_name
        self.encoder_module = ENCODER_REGISTRY[config_name](k_dim)
        self.use_pos_enc = (config_name != "w/o PosEnc")
        self.pos_enc = SinusoidalPositionalEncoding(N_TOKENS, D_MODEL)
        self.transformer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS, dim_feedforward=D_FF,
            activation="relu", batch_first=True, norm_first=True, dropout=0.0)
        self.classifier = nn.Linear(D_MODEL, n_classes)

    def forward(self, tokens):
        x = self.encoder_module(tokens)
        if self.use_pos_enc:
            x = self.pos_enc(x)
        x = self.transformer(x)
        return self.classifier(x.mean(dim=1))

print("Cell 2 loaded: all 5 ablation configs registered — "
      f"{list(ENCODER_REGISTRY.keys())}")

Cell 2 loaded: all 5 ablation configs registered — ['Full', 'w/o QNN', 'w/o Bottleneck', '1-Qubit', 'w/o PosEnc']


In [4]:
# CELL 3 — Training loop + evaluation, works for any AblationQuantFormer config

def get_param_groups(model):
    decay, no_decay = [], []
    for name, param in model.named_parameters():
        (no_decay if "q_layer" in name else decay).append(param)
    return [{"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0}]

def train_config(seed, k_dim, n_classes, config_name, train_tokens, train_labels,
                  val_tokens, val_labels):
    torch.manual_seed(seed)
    model = AblationQuantFormer(k_dim, n_classes, config_name).to(DEVICE)
    optimizer = torch.optim.Adam(get_param_groups(model), lr=LR)
    criterion = nn.CrossEntropyLoss()
    n_train = train_tokens.shape[0]
    best_val_acc, best_state = -1, None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train)
        for i in range(0, n_train, TRAIN_BATCH_SIZE):
            idx = perm[i:i + TRAIN_BATCH_SIZE]
            xb, yb = train_tokens[idx].to(DEVICE), train_labels[idx].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_acc = accuracy_score(val_labels, model(val_tokens.to(DEVICE)).argmax(dim=1).cpu())
        if val_acc > best_val_acc:
            best_val_acc, best_state = val_acc, {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

def evaluate_with_predictions(model, test_tokens, test_labels, n_classes):
    model.eval()
    with torch.no_grad():
        preds = model(test_tokens.to(DEVICE)).argmax(dim=1).cpu().numpy()
    labels_np = test_labels.numpy()
    cm = confusion_matrix(labels_np, preds, labels=list(range(n_classes)))
    per_class_acc = cm.diagonal() / cm.sum(axis=1).clip(min=1)
    metrics = {
        "OA": accuracy_score(labels_np, preds), "AA": per_class_acc.mean(),
        "kappa": cohen_kappa_score(labels_np, preds), "MCC": matthews_corrcoef(labels_np, preds),
    }
    return metrics, preds  # keep raw preds for McNemar

print("Cell 3 loaded: train_config() / evaluate_with_predictions() ready.")

Cell 3 loaded: train_config() / evaluate_with_predictions() ready.


In [5]:
# CELL 4 — Full ablation matrix: 5 datasets x 5 configs x 3 seeds = 75 runs
CONFIGS = ["Full", "w/o QNN", "w/o Bottleneck", "w/o PosEnc", "1-Qubit"]
ablation_results = {}      # metrics
ablation_predictions = {}  # raw predictions, for McNemar

for name in ABLATION_DATASETS:
    ds = datasets[name]
    ablation_results[name] = {c: [] for c in CONFIGS}
    ablation_predictions[name] = {c: {} for c in CONFIGS}

    for config in CONFIGS:
        for seed in SEEDS:
            print(f"\n=== {name} [{config}], seed={seed} ===")
            start = time.time()
            model = train_config(seed, ds["k_dim"], ds["n_classes"], config,
                                  ds["train_tokens"], ds["train_labels"],
                                  ds["val_tokens"], ds["val_labels"])
            elapsed = time.time() - start
            metrics, preds = evaluate_with_predictions(model, ds["test_tokens"], ds["test_labels"], ds["n_classes"])
            metrics["seed"] = seed
            metrics["train_time_sec"] = elapsed
            ablation_results[name][config].append(metrics)
            ablation_predictions[name][config][seed] = preds.tolist()
            print(f"  OA={metrics['OA']:.4f} AA={metrics['AA']:.4f} "
                  f"kappa={metrics['kappa']:.4f} MCC={metrics['MCC']:.4f} ({elapsed/60:.1f} min)")

with open("phase5_ablation_results.json", "w") as f:
    json.dump(ablation_results, f, indent=2)
with open("phase5_ablation_predictions.json", "w") as f:
    json.dump(ablation_predictions, f, indent=2)
print("\nSaved phase5_ablation_results.json and phase5_ablation_predictions.json")


=== IndianPines [Full], seed=42 ===
  OA=0.8739 AA=0.8002 kappa=0.8561 MCC=0.8566 (1.8 min)

=== IndianPines [Full], seed=43 ===
  OA=0.9007 AA=0.8261 kappa=0.8867 MCC=0.8869 (1.7 min)

=== IndianPines [Full], seed=44 ===
  OA=0.8827 AA=0.7519 kappa=0.8661 MCC=0.8664 (1.7 min)

=== IndianPines [w/o QNN], seed=42 ===
  OA=0.9201 AA=0.8945 kappa=0.9089 MCC=0.9093 (0.1 min)

=== IndianPines [w/o QNN], seed=43 ===
  OA=0.9298 AA=0.8767 kappa=0.9199 MCC=0.9201 (0.1 min)

=== IndianPines [w/o QNN], seed=44 ===
  OA=0.9107 AA=0.8668 kappa=0.8984 MCC=0.8988 (0.1 min)

=== IndianPines [w/o Bottleneck], seed=42 ===
  OA=0.9589 AA=0.9420 kappa=0.9531 MCC=0.9532 (0.1 min)

=== IndianPines [w/o Bottleneck], seed=43 ===
  OA=0.9627 AA=0.9431 kappa=0.9575 MCC=0.9575 (0.1 min)

=== IndianPines [w/o Bottleneck], seed=44 ===
  OA=0.9595 AA=0.9544 kappa=0.9538 MCC=0.9538 (0.1 min)

=== IndianPines [w/o PosEnc], seed=42 ===
  OA=0.9271 AA=0.8209 kappa=0.9169 MCC=0.9171 (1.7 min)

=== IndianPines [w/o Pos

In [8]:
# CELL 5 — Statistical comparison: Full vs. each ablation config (p-value clipping fix applied)
from statsmodels.stats.multitest import multipletests

test_labels_by_dataset = {name: datasets[name]["test_labels"].numpy() for name in ABLATION_DATASETS}
comparison_configs = ["w/o QNN", "w/o Bottleneck", "w/o PosEnc", "1-Qubit"]

MIN_PVALUE = 1e-300  # floor to avoid log(0) = -inf in Fisher's method

final_stats = {}
for name in ABLATION_DATASETS:
    final_stats[name] = {}
    true_labels = test_labels_by_dataset[name]

    for comp_config in comparison_configs:
        per_seed_pvalues = []
        for seed in SEEDS:
            full_preds = np.array(ablation_predictions[name]["Full"][seed])
            comp_preds = np.array(ablation_predictions[name][comp_config][seed])

            full_correct = (full_preds == true_labels)
            comp_correct = (comp_preds == true_labels)

            both_correct = np.sum(full_correct & comp_correct)
            full_only = np.sum(full_correct & ~comp_correct)
            comp_only = np.sum(~full_correct & comp_correct)
            neither = np.sum(~full_correct & ~comp_correct)
            table = [[both_correct, full_only], [comp_only, neither]]

            result = mcnemar(table, exact=False, correction=True)
            # clip away from exact 0 immediately, before any downstream use
            per_seed_pvalues.append(max(result.pvalue, MIN_PVALUE))

        final_stats[name][comp_config] = {"per_seed_pvalues": per_seed_pvalues}

    # BH correction within each seed, across the 4 comparisons
    for seed_idx, seed in enumerate(SEEDS):
        pvals_this_seed = [final_stats[name][c]["per_seed_pvalues"][seed_idx] for c in comparison_configs]
        _, corrected, _, _ = multipletests(pvals_this_seed, method="fdr_bh")
        # clip again post-correction, since BH can also push a value to exactly 0 in edge cases
        corrected = [max(p, MIN_PVALUE) for p in corrected]
        for c, corr_p in zip(comparison_configs, corrected):
            final_stats[name][c].setdefault("bh_corrected_pvalues", []).append(corr_p)

    # Combine the 3 seeds' BH-corrected p-values via Fisher's method
    for comp_config in comparison_configs:
        combined_stat, combined_p = combine_pvalues(
            final_stats[name][comp_config]["bh_corrected_pvalues"], method="fisher"
        )
        final_stats[name][comp_config]["combined_pvalue"] = combined_p
        final_stats[name][comp_config]["significant"] = bool(combined_p < 0.05)

# --- Effect size alongside significance, so tiny-but-significant isn't mistaken for meaningful ---
print(f"\n{'='*90}\nPHASE 5 — McNEMAR SUMMARY (Full vs. each config, Fisher-combined, p-value floor applied)\n{'='*90}")
for name in ABLATION_DATASETS:
    print(f"\n{name}:")
    for comp_config in comparison_configs:
        s = final_stats[name][comp_config]
        oa_full = np.mean([r["OA"] for r in ablation_results[name]["Full"]])
        oa_comp = np.mean([r["OA"] for r in ablation_results[name][comp_config]])
        effect = (oa_comp - oa_full) * 100  # percentage points, signed: positive = comp_config wins
        magnitude = "negligible" if abs(effect) < 0.5 else ("moderate" if abs(effect) < 3 else "large")
        print(f"  Full ({oa_full:.4f}) vs {comp_config} ({oa_comp:.4f}): "
              f"effect={effect:+.2f}pp [{magnitude}] "
              f"combined p={s['combined_pvalue']:.4g} "
              f"{'[SIGNIFICANT]' if s['significant'] else '[not significant]'}")

with open("phase5_mcnemar_results.json", "w") as f:
    json.dump(final_stats, f, indent=2)
print("\nSaved phase5_mcnemar_results.json")


PHASE 5 — McNEMAR SUMMARY (Full vs. each config, Fisher-combined, p-value floor applied)

IndianPines:
  Full (0.8858) vs w/o QNN (0.9202): effect=+3.44pp [large] combined p=1.538e-50 [SIGNIFICANT]
  Full (0.8858) vs w/o Bottleneck (0.9604): effect=+7.46pp [large] combined p=1.169e-274 [SIGNIFICANT]
  Full (0.8858) vs w/o PosEnc (0.9225): effect=+3.67pp [large] combined p=2.894e-68 [SIGNIFICANT]
  Full (0.8858) vs 1-Qubit (0.6776): effect=-20.82pp [large] combined p=0 [SIGNIFICANT]

PaviaUniversity:
  Full (0.9892) vs w/o QNN (0.9890): effect=-0.02pp [negligible] combined p=9.892e-27 [SIGNIFICANT]
  Full (0.9892) vs w/o Bottleneck (0.9937): effect=+0.46pp [negligible] combined p=6.557e-54 [SIGNIFICANT]
  Full (0.9892) vs w/o PosEnc (0.9825): effect=-0.67pp [moderate] combined p=7.019e-67 [SIGNIFICANT]
  Full (0.9892) vs 1-Qubit (0.9087): effect=-8.04pp [large] combined p=0 [SIGNIFICANT]

Salinas:
  Full (0.9928) vs w/o QNN (0.9900): effect=-0.28pp [negligible] combined p=8.634e-36 [SI